# Build the FAST Utilities APK in Google Colab

This notebook builds an **installable release APK** for the FAST Utilities Android app using the
Android Gradle Plugin — **no Android Studio and no Expo account / EAS login required**.

**What it does:**
1. Installs **JDK 17**, **Node 20** and the **Android SDK** (API 36, NDK 27.1, CMake 3.30.5).
2. Clones this repo and installs its dependencies.
3. Runs `expo prebuild` to generate the native `android/` project.
4. Builds `app-release.apk` with Gradle.
5. Downloads the APK **and a full build log** to your machine.

**Notes**
- The first build takes **~10–20 minutes** (NDK + Gradle downloads). Later runs are faster.
- The APK is signed with the Expo **debug keystore** by default, so it is fine to **sideload /
  install directly** on a device, but it is **not** for Google Play. For a Play Store AAB you must
  configure a release keystore (see the README) or use `eas build -p android --profile production`.
- **If anything fails**, every step writes to `/content/fast_utilities_build.log` — run the final
  cell to download it and share it for debugging.
- Run the cells **top to bottom**. If a cell fails, re-run it.



In [ ]:
# ── 0. Start a persistent build log (every step appends here) ──
import os, datetime

LOG = "/content/fast_utilities_build.log"

def log_header():
    with open(LOG, "a") as f:
        f.write("=" * 70 + "\n")
        f.write("FAST Utilities build log\n")
        f.write("Started: " + datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + "\n")
        f.write("=" * 70 + "\n")

log_header()
print("Log file:", LOG)


In [ ]:
# ── 1. Install JDK 17 (required by Android Gradle Plugin 8.12 / Kotlin 2.1) ──
!apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless unzip curl 2>&1 | tee -a $LOG

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

!$JAVA_HOME/bin/java -version 2>&1 | tee -a $LOG


In [ ]:
# ── 2. Install Node.js 20 (Expo SDK 57 requires Node 20+) ──
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>&1 | tee -a $LOG
!apt-get install -y -qq nodejs 2>&1 | tee -a $LOG

!node --version 2>&1 | tee -a $LOG
!npm --version 2>&1 | tee -a $LOG


In [ ]:
# ── 3. Install the Android SDK and the exact packages this project pins ──
# Versions come from node_modules/react-native/ReactAndroid/gradle/libs.versions.toml:
#   compileSdk/targetSdk = 36, build-tools = 36.0.0, ndk = 27.1.12297006, cmake = 3.30.5
import os
os.environ["ANDROID_HOME"] = "/opt/android-sdk"
os.environ["ANDROID_SDK_ROOT"] = "/opt/android-sdk"

!mkdir -p $ANDROID_HOME/cmdline-tools
!curl -fsSL -o /tmp/cmdtools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip 2>&1 | tee -a $LOG
!unzip -q -o /tmp/cmdtools.zip -d $ANDROID_HOME/cmdline-tools
!mv $ANDROID_HOME/cmdline-tools/cmdline-tools $ANDROID_HOME/cmdline-tools/latest

# Accept all licenses, then install the SDK components (NDK is the big one, ~2–3 min)
!yes | $ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null 2>&1
!$ANDROID_HOME/cmdline-tools/latest/bin/sdkmanager \
    "platform-tools" \
    "platforms;android-36" \
    "build-tools;36.0.0" \
    "ndk;27.1.12297006" \
    "cmake;3.30.5" 2>&1 | tee -a $LOG


In [ ]:
# ── 4. Clone this repo and install dependencies ──
!rm -rf /content/fast-utilities-android-v1
!git clone --depth 1 https://github.com/ammarasad2005/fast-utilities-android-v1.git /content/fast-utilities-android-v1 2>&1 | tee -a $LOG

%cd /content/fast-utilities-android-v1
!npm ci 2>&1 | tee -a $LOG


In [ ]:
# ── 5. Generate the native Android project (expo prebuild) ──
%cd /content/fast-utilities-android-v1
!npx expo prebuild --platform android --no-install 2>&1 | tee -a $LOG

# Point Gradle at the SDK + JDK we installed and give it enough memory
!echo "sdk.dir=/opt/android-sdk" > android/local.properties
!echo "org.gradle.java.home=/usr/lib/jvm/java-17-openjdk-amd64" >> android/gradle.properties
!echo "org.gradle.jvmargs=-Xmx4096m -XX:MaxMetaspaceSize=1024m" >> android/gradle.properties

!echo "--- generated android/ folder ---" | tee -a $LOG
!ls android 2>&1 | tee -a $LOG


In [ ]:
# ── 5b. DIAGNOSTIC — run the exact autolinking node command Gradle runs ──
# This isolates whether the failure is in the node autolinking step vs. Gradle.
# If this prints an error, we know node/autolinking is the problem and see WHY.
%cd /content/fast-utilities-android-v1/android

import subprocess, sys
cmd = [
  "node", "--no-warnings", "--eval", "require('expo/bin/autolinking')",
  "expo-modules-autolinking", "react-native-config",
  "--platform", "android", "--json",
  "--project-root", "/content/fast-utilities-android-v1",
  "--source-dir", "/content/fast-utilities-android-v1/android",
]
print("Running:", " ".join(cmd))
p = subprocess.run(cmd, capture_output=True, text=True)
with open("/content/fast_utilities_build.log", "a") as f:
    f.write("--- autolinking diagnostic ---\n")
    f.write("exit code: " + str(p.returncode) + "\n")
    f.write("STDOUT (last 800 chars):\n" + p.stdout[-800:] + "\n")
    if p.stderr:
        f.write("STDERR:\n" + p.stderr + "\n")
print("Autolinking exit code:", p.returncode)
if p.returncode != 0:
    print("STDERR:", p.stderr)


In [ ]:
# ── 6. Build the release APK (10–25 min on first run) ──
# --stacktrace --info makes Gradle print the ACTUAL node command + its error
# instead of just "Process 'command 'node'' finished with non-zero exit value 1".
%cd /content/fast-utilities-android-v1/android

import subprocess, sys
p = subprocess.Popen(
    ["./gradlew", "assembleRelease", "--no-daemon", "--stacktrace", "--info"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
with open("/content/fast_utilities_build.log", "a") as log:
    for line in p.stdout:
        sys.stdout.write(line)
        log.write(line)
p.wait()
print("\nGradle exit code:", p.returncode)


In [ ]:
# ── 7. Download the APK AND the build log ──
import os
from google.colab import files

apk = "/content/fast-utilities-android-v1/android/app/build/outputs/apk/release/app-release.apk"
log = "/content/fast_utilities_build.log"

print("APK exists:", os.path.exists(apk))
if os.path.exists(apk):
    print("APK size: %.1f MB" % (os.path.getsize(apk) / 1e6))
print("Log exists:", os.path.exists(log))
if os.path.exists(log):
    print("Log size: %.1f KB" % (os.path.getsize(log) / 1e3))

# Download BOTH files — share the .log if the build failed.
if os.path.exists(apk):
    files.download(apk)
files.download(log)

# Optional: also save copies to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "$apk" /content/drive/MyDrive/fast-utilities-v1.apk
# !cp "$log" /content/drive/MyDrive/fast_utilities_build.log


## If the build failed

Run the cell above to download **`fast_utilities_build.log`** and share it. The log captures every
step (toolchain installs, clone, dependency install, prebuild, and the full Gradle output), which is
usually enough to diagnose the exact failure.

Common issues and where to look in the log:
- **`Could not resolve` / dependency download errors** → network hiccup; re-run the failing cell.
- **`Installed Build Tools revision ... is corrupted`** → re-run cell 3 (SDK install).
- **`Failed to apply plugin 'com.android...' / requires Java`** → the JDK wasn't picked up; check the
  `java -version` line in the log is 17.
- **`ERROR: ... NDK ... not found`** → re-run cell 3 and confirm the NDK line succeeded.
